# FineTuning FaceNet part 2

## Installing Dependencies

In [1]:
!pip uninstall -f pillow
!pip install pillow --upgrade
!pip install facenet-pytorch==2.5.3 --upgrade


Usage:   
  pip3 uninstall [options] <package> ...
  pip3 uninstall [options] -r <requirements file> ...

no such option: -f


## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from scipy.stats import mode
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from facenet_pytorch import InceptionResnetV1

## Configurations

In [ ]:
class Config:
    BASE_DIR = "/kaggle/input/surveillance-for-retail-stores/face_identification/face_identification"
    TRAIN_CSV = os.path.join(BASE_DIR, "trainset.csv")
    CHECKPOINT_DIR = "/kaggle/working/checkpoints/facenet_triplet"
    BATCH_SIZE = 512
    NUM_EPOCHS = 34  # Total epochs (24 + 10 for fine-tuning)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EMBEDDING_DIM = 1024
    IMAGE_SIZE = (160, 160)
    NUM_WORKERS = 2
    SCALING_FACTOR = 1.0
    NUM_CLASSES = None
    START_EPOCH = 24  # Start from epoch 24 checkpoint
    CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_model_epoch_24.pth")

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
print(f"Using device: {Config.DEVICE}")

## Dataset Helper Class

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        try:
            img_path = self.df.iloc[idx]['full_path']
            label = self.df.iloc[idx]['label']
            image = Image.open(img_path).convert('RGB')
            image = np.array(image)
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, label
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return self.__getitem__((idx + 1) % len(self.df))

## Model Definition + Triplet Loss

In [ ]:
class FaceNetModel(nn.Module):
    def __init__(self, embedding_dim=1024):
        super(FaceNetModel, self).__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2')
        self.backbone.classify = False
        self.embedding_layer = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
    
    def forward(self, x, return_embeddings=True):
        features = self.backbone(x)
        embeddings = self.embedding_layer(features)
        scaled_embeddings = embeddings * Config.SCALING_FACTOR
        return scaled_embeddings

class TripletLoss(nn.Module):
    def __init__(self, margin):
        super(TripletLoss, self).__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        anchor = F.normalize(anchor, p=2, dim=1)
        positive = F.normalize(positive, p=2, dim=1)
        negative = F.normalize(negative, p=2, dim=1)
        pos_dist = F.pairwise_distance(anchor, positive)
        neg_dist = F.pairwise_distance(anchor, negative)
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean(), pos_dist.mean(), neg_dist.mean()

def get_margin(epoch):
    # Fixed margin of 2.0 for fine-tuning from epoch 24 onward
    return 2.0 if epoch >= Config.START_EPOCH else 1.6

def generate_triplets(embeddings, labels, margin, epoch):
    labels = labels.detach()
    anchor, positive, negative = [], [], []
    batch_size = embeddings.size(0)
    dist_matrix = torch.cdist(embeddings, embeddings)
    hard_ratio = min(0.05 * epoch, 0.9)  # Increased to 0.05, cap at 0.9
    
    for i in range(batch_size):
        label = labels[i]
        pos_mask = (labels == label) & (torch.arange(batch_size).to(Config.DEVICE) != i)
        neg_mask = (labels != label)
        
        pos_indices = torch.where(pos_mask)[0]
        neg_indices = torch.where(neg_mask)[0]
        
        if len(pos_indices) == 0 or len(neg_indices) == 0:
            continue
        
        pos_dists = dist_matrix[i, pos_indices]
        neg_dists = dist_matrix[i, neg_indices]
        
        pos_idx = pos_indices[torch.argmax(pos_dists)]  # Hardest positive
        pos_dist = pos_dists.max()
        
        semi_hard_mask = (neg_dists > pos_dist) & (neg_dists < pos_dist + margin)
        semi_hard_indices = neg_indices[semi_hard_mask]
        if len(semi_hard_indices) > 0 and torch.rand(1).item() >= hard_ratio:
            neg_idx = semi_hard_indices[torch.randint(0, len(semi_hard_indices), (1,)).item()]
        else:
            neg_idx = neg_indices[torch.argmin(neg_dists)]  # Hardest negative
        
        anchor.append(embeddings[i])
        positive.append(embeddings[pos_idx])
        negative.append(embeddings[neg_idx])
    
    if len(anchor) == 0:
        print(f"Warning: No triplets generated in epoch {epoch+1}. All samples may have the same label.")
        return None, None, None
    
    print(f"Epoch {epoch+1}: Generated {len(anchor)} triplets")
    return (torch.stack(anchor), torch.stack(positive), torch.stack(negative))

def compute_class_weights(df):
    class_counts = df['label'].value_counts().sort_index()
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[df['label']].values
    print(f"Class Weights - Min: {class_weights.min():.4f}, Max: {class_weights.max():.4f}")
    return sample_weights

## Validation with clustering accuracy

In [ ]:
def evaluate_embeddings(model, val_loader, criterion_triplet):
    model.eval()
    all_embeddings = []
    all_labels = []
    val_triplet_loss_total = 0.0
    num_val_batches = 0
    val_criterion = TripletLoss(margin=0.8)  # Fixed margin for validation
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Generating validation embeddings"):
            images = images.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            embeddings = model(images)
            anchor, positive, negative = generate_triplets(embeddings, labels, criterion_triplet.margin, 0)
            if anchor is not None:
                val_triplet_loss = val_criterion(anchor, positive, negative)[0]
                val_triplet_loss_total += val_triplet_loss.item()
                num_val_batches += 1
            all_embeddings.append(embeddings.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_embeddings = np.concatenate(all_embeddings)
    all_labels = np.concatenate(all_labels)
    avg_val_triplet_loss = val_triplet_loss_total / num_val_batches if num_val_batches > 0 else 0.0
    
    embeddings_norm = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)
    dist_matrix = torch.cdist(torch.tensor(embeddings_norm), torch.tensor(embeddings_norm)).numpy()
    pos_mask = (all_labels[:, None] == all_labels[None, :]) & ~np.eye(len(all_labels), dtype=bool)
    neg_mask = all_labels[:, None] != all_labels[None, :]
    pos_dists = dist_matrix[pos_mask]
    neg_dists = dist_matrix[neg_mask]
    
    mean_pos_dist = np.mean(pos_dists) if len(pos_dists) > 0 else 0.0
    mean_neg_dist = np.mean(neg_dists) if len(neg_dists) > 0 else float('nan')
    max_pos_dist = np.max(pos_dists) if len(pos_dists) > 0 else 0.0
    
    kmeans = KMeans(n_clusters=Config.NUM_CLASSES, random_state=42, n_init=10).fit(embeddings_norm)
    pred_labels = kmeans.labels_
    label_map = {}
    for cluster in range(Config.NUM_CLASSES):
        cluster_labels = all_labels[pred_labels == cluster]
        if len(cluster_labels) > 0:
            mode_result = mode(cluster_labels)
            label_map[cluster] = mode_result.mode.item()
    mapped_labels = [label_map.get(pred, -1) for pred in pred_labels]
    accuracy = np.mean([1 if pred == true else 0 for pred, true in zip(mapped_labels, all_labels)]) * 100
    
    # Per-class accuracy
    per_class_acc = {}
    for true_label in range(Config.NUM_CLASSES):
        class_mask = (all_labels == true_label)
        if class_mask.sum() > 0:
            class_correct = sum(1 for pred, true in zip(mapped_labels, all_labels) if pred == true and true == true_label)
            per_class_acc[true_label] = (class_correct / class_mask.sum()) * 100
    print("Per-Class Accuracy:", {k: f"{v:.2f}%" for k, v in per_class_acc.items()})
    
    print(f"Validation - Mean Pos Dist: {mean_pos_dist:.4f}, Mean Neg Dist: {mean_neg_dist:.4f}, "
          f"Max Pos Dist: {max_pos_dist:.4f}, Clustering Accuracy: {accuracy:.2f}%, "
          f"Norm Embedding Std: {np.std(embeddings_norm):.4f}, Raw Embedding Std: {np.std(all_embeddings):.4f}, "
          f"Avg Val Triplet Loss: {avg_val_triplet_loss:.4f}")
    return mean_pos_dist, mean_neg_dist, max_pos_dist, accuracy, avg_val_triplet_loss

## Training

In [ ]:
def train_step(model, images, labels, epoch, batch_idx, optimizer, criterion_triplet, train_loader):
    optimizer.zero_grad()
    embeddings = model(images)
    anchor, positive, negative = generate_triplets(embeddings, labels, criterion_triplet.margin, epoch)
    if anchor is None:
        return 0.0, 0.0, 0.0
    triplet_loss, mean_pos_dist, mean_neg_dist = criterion_triplet(anchor, positive, negative)
    l2_reg = 0.001 * torch.norm(embeddings, p=2)
    pos_target = max(0.8 - 0.03 * epoch, 0.4)
    pos_penalty = 0.5 * (F.relu(mean_pos_dist - pos_target) + F.relu(0.05 - mean_pos_dist))
    loss = triplet_loss + l2_reg + pos_penalty
    if batch_idx % 10 == 0 or batch_idx == len(train_loader) - 1:
        print(f"Epoch {epoch+1}, Batch {batch_idx} - Triplet Loss: {triplet_loss.item():.4f}, "
              f"Mean Pos Dist: {mean_pos_dist:.4f}, Mean Neg Dist: {mean_neg_dist:.4f}, "
              f"Embeddings Mean: {embeddings.mean():.4f}, Std: {embeddings.std():.4f}")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()
    return loss.item(), mean_pos_dist.item(), mean_neg_dist.item()

def train_model(model, train_loader, val_loader, criterion_triplet, optimizer, scheduler, start_epoch=Config.START_EPOCH, num_epochs=Config.NUM_EPOCHS):
    best_acc = 98.90  # Starting from epoch 24's accuracy
    best_epoch = Config.START_EPOCH
    prev_neg_dist = 0.0
    plateau_count = 0
    
    # Load the epoch 24 checkpoint
    if os.path.exists(Config.CHECKPOINT_PATH):
        checkpoint = torch.load(Config.CHECKPOINT_PATH, weights_only=True)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded checkpoint from {Config.CHECKPOINT_PATH} (epoch {Config.START_EPOCH})")
    else:
        raise FileNotFoundError(f"Checkpoint {Config.CHECKPOINT_PATH} not found!")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        train_loss = 0.0
        total_pos_dist = 0.0
        total_neg_dist = 0.0
        
        criterion_triplet.margin = get_margin(epoch)
        print(f"Epoch {epoch+1} - Margin: {criterion_triplet.margin:.2f}")
        
        for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training")):
            images = images.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            loss, pos_dist, neg_dist = train_step(model, images, labels, epoch, batch_idx, optimizer, criterion_triplet, train_loader)
            train_loss += loss
            total_pos_dist += pos_dist
            total_neg_dist += neg_dist
        
        train_loss /= len(train_loader)
        avg_pos_dist = total_pos_dist / len(train_loader)
        avg_neg_dist = total_neg_dist / len(train_loader)
        print(f"Epoch {epoch+1} Summary - Train Loss: {train_loss:.4f}, "
              f"Avg Pos Dist: {avg_pos_dist:.4f}, Avg Neg Dist: {avg_neg_dist:.4f}")
        
        mean_pos_dist, mean_neg_dist, max_pos_dist, val_acc, val_triplet_loss = evaluate_embeddings(model, val_loader, criterion_triplet)
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            torch.save({'model_state_dict': model.state_dict()}, 
                       f"{Config.CHECKPOINT_DIR}/best_model_epoch_{best_epoch}.pth")
            print(f"Saved best model at epoch {best_epoch} with Clustering Accuracy: {val_acc:.2f}%")
        
        scheduler.step()  # CosineAnnealingLR steps every epoch
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        if not np.isnan(mean_neg_dist) and abs(mean_neg_dist - prev_neg_dist) < 0.01:
            plateau_count += 1
            if plateau_count >= 7:
                print(f"Plateau detected for {plateau_count} epochs")
        else:
            plateau_count = 0
        prev_neg_dist = mean_neg_dist if not np.isnan(mean_neg_dist) else prev_neg_dist
        
        if epoch > start_epoch + 6 and val_acc < 99.0 and plateau_count >= 7:
            print(f"Early stopping at epoch {epoch+1}: Accuracy plateaued at {val_acc:.2f}%")
            break
    
    return best_acc, best_epoch

## Main

In [22]:


def main():
    torch.manual_seed(42)
    np.random.seed(42)
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

    print("Loading training data...")
    train_df = pd.read_csv(Config.TRAIN_CSV, sep=',', engine='python', header=0, 
                          names=['image_path', 'gt'], on_bad_lines='skip')
    train_df = train_df.dropna()
    train_df['image_path'] = train_df['image_path'].str.strip()
    train_df['gt'] = train_df['gt'].str.strip()
    train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(Config.BASE_DIR, x))
    le = LabelEncoder()
    train_df['label'] = le.fit_transform(train_df['gt'])
    Config.NUM_CLASSES = len(le.classes_)
    print(f"Training data loaded: {len(train_df)} samples, {Config.NUM_CLASSES} classes")
    print(f"Class distribution: {train_df['label'].value_counts().to_dict()}")

    train_data, val_data = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)
    print(f"Train split: {len(train_data)} samples, Val split: {len(val_data)} samples")

    train_transform = A.Compose([
        A.Resize(*Config.IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Rotate(limit=10, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2, p=0.5),
        A.CoarseDropout(max_holes=4, max_height=4, max_width=4, min_holes=1, min_height=2, min_width=2, p=0.3),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])
    val_transform = A.Compose([
        A.Resize(*Config.IMAGE_SIZE),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

    train_dataset = FaceDataset(train_data, transform=train_transform)
    val_dataset = FaceDataset(val_data, transform=val_transform)

    sample_weights = compute_class_weights(train_data)
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, 
                             num_workers=Config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, 
                           num_workers=Config.NUM_WORKERS, pin_memory=True)

    model = FaceNetModel(embedding_dim=Config.EMBEDDING_DIM).to(Config.DEVICE)
    criterion_triplet = TripletLoss(margin=2.0)  # Initial margin for fine-tuning
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)  # 10 epochs decay

    best_acc, best_epoch = train_model(model, train_loader, val_loader, criterion_triplet, optimizer, scheduler)

    print("Final embedding evaluation:")
    checkpoint = torch.load(f"{Config.CHECKPOINT_DIR}/best_model_epoch_{best_epoch}.pth", weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    mean_pos_dist, mean_neg_dist, max_pos_dist, final_acc, _ = evaluate_embeddings(model, val_loader, criterion_triplet)
    print(f"Final Results - Mean Pos Dist: {mean_pos_dist:.4f}, Mean Neg Dist: {mean_neg_dist:.4f}, "
          f"Max Pos Dist: {max_pos_dist:.4f}, Clustering Accuracy: {final_acc:.2f}%")
    
    return model, best_acc, le, train_df

if __name__ == "__main__":
    model, best_acc, le, train_df = main()

Using device: cuda
Loading training data...
Training data loaded: 6828 samples, 125 classes
Class distribution: {115: 283, 21: 245, 36: 203, 10: 191, 112: 163, 18: 110, 92: 98, 60: 98, 107: 98, 65: 96, 83: 94, 54: 91, 32: 86, 89: 85, 117: 84, 98: 80, 14: 79, 3: 77, 104: 77, 71: 76, 20: 75, 47: 73, 69: 73, 84: 72, 41: 71, 102: 70, 121: 69, 31: 69, 28: 68, 33: 68, 35: 68, 124: 68, 85: 67, 66: 67, 96: 67, 88: 64, 40: 63, 30: 63, 29: 63, 123: 62, 93: 62, 72: 62, 52: 60, 120: 60, 16: 59, 67: 56, 101: 56, 51: 56, 59: 55, 57: 55, 46: 55, 109: 54, 114: 54, 105: 54, 48: 54, 4: 53, 106: 53, 17: 52, 42: 52, 61: 51, 6: 50, 8: 49, 100: 49, 97: 49, 118: 48, 53: 48, 80: 47, 90: 46, 116: 46, 62: 45, 87: 43, 91: 42, 64: 42, 77: 41, 73: 39, 74: 38, 43: 37, 111: 37, 49: 37, 63: 37, 5: 36, 22: 35, 12: 35, 55: 35, 9: 33, 99: 33, 81: 33, 122: 32, 108: 32, 75: 32, 78: 32, 76: 31, 1: 31, 24: 30, 113: 30, 79: 30, 94: 29, 119: 29, 13: 26, 44: 26, 50: 25, 23: 25, 25: 24, 26: 24, 19: 23, 11: 22, 7: 21, 56: 21, 95

/usr/local/lib/python3.10/dist-packages/facenet_pytorch/models/inception_resnet_v1.py:329: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)

Loaded checkpoint from /kaggle/working/checkpoints/facenet_triplet/best_model_epoch_24.pth (epoch 24)
Epoch 25 - Margin: 2.00


Epoch 25/34 - Training:   0%|          | 0/11 [00:00<?, ?it/s]

Epoch 25: Generated 505 triplets
Epoch 25, Batch 0 - Triplet Loss: 1.2582, Mean Pos Dist: 0.3599, Mean Neg Dist: 1.1017, Embeddings Mean: 0.0003, Std: 0.9790


...

Generating validation embeddings:  67%|██████▋   | 2/3 [00:03<00:01,  1.52s/it]

Epoch 1: Generated 497 triplets


Generating validation embeddings: 100%|██████████| 3/3 [00:03<00:00,  1.31s/it]

Epoch 1: Generated 306 triplets


Per-Class Accuracy: {0: '100.00%', 1: '100.00%', 2: '100.00%', 3: '100.00%', 4: '100.00%', 5: '100.00%', 6: '100.00%', 7: '100.00%', 8: '100.00%', 9: '100.00%', 10: '100.00%', 11: '100.00%', 12: '100.00%', 13: '100.00%', 14: '93.75%', 15: '100.00%', 16: '100.00%', 17: '100.00%', 18: '100.00%', 19: '100.00%', 20: '100.00%', 21: '100.00%', 22: '100.00%', 23: '100.00%', 24: '100.00%', 25: '100.00%', 26: '100.00%', 27: '0.00%', 28: '100.00%', 29: '100.00%', 30: '100.00%', 31: '100.00%', 32: '100.00%', 33: '100.00%', 34: '100.00%', 35: '100.00%', 36: '100.00%', 37: '100.00%', 38: '100.00%', 39: '100.00%', 40: '100.00%', 41: '100.00%', 42: '100.00%', 43: '100.00%', 44: '100.00%', 45: '0.00%', 46: '100.00%', 47: '100.00%', 48: '100.00%', 49: '100.00%', 50: '100.00%', 51: '100.00%', 52: '100.00%', 53: '100.00%', 54: '100.00%', 55: '100.00%', 56: '100.00%', 57: '100.00%', 58: '66.67%', 59: '100.00%', 60: '100.00%', 61: '100.00%', 62: '100.00%', 63: '100.00%', 64: '100.00%', 65: '100.00%', 66: '